# Setup Build Waves Job

Patches the bundle-deployed build jobs so their Genie Code tasks are real `genie_task`s. Runs as the setup job after **every** `bundle deploy`, because deploy restores the placeholders. Idempotent.

The build job is declared in `build_waves.job.yml` with a single placeholder task. This notebook:

1. **Publishes UC Skills** from `skills_dir/` to `{catalog}.{schema}.{skill_id}` (create, upload via Files API, finalize).
2. **Creates or updates Genie Code automations** (one per role: runner, integrator, smoke, capability).
3. **Reads `waves.json`** and dynamically generates the full task graph: per wave, a `for_each` of session runners at full wave width, then an integrator, a gate notebook and an If/else `gate_check` condition task. Genie Code tasks get automatic retries (`task_retries`).
4. **Resets the build job** via `jobs/reset` with the generated task graph (replacing the single placeholder). All Jobs calls use API 2.2 (`jobs_api.py`): 2.1 rejects jobs with more than 100 tasks, and every `jobs/get` / `runs/get` read follows `next_page_token` so no task is lost.
5. **Patches the smoke job** the same way (2-item for_each nonce echo + a read-only `capability_probe` that checks CLI, apps/jobs read, SQL and telemetry access for converge sessions).
6. Optionally **runs the smoke job** and checks the output.

Before generating anything it **validates the manifest**: unique ids, valid modes, defined `runtime_targets` (`app`, `job`, `pipeline`, `dashboard`, `genie_space`, `table`), well-formed `validate_bundles` entries, and the single-writer rule (a wave with a `converge` or `allow_deploy` session must have exactly one session; its concurrency is forced to 1). Waves with `"archived": true` (earlier builds) keep the id and runtime-target checks, skip the single-writer rule, and generate no tasks; their sessions count as done for `depends_on`. Validation and task generation live in `waves_manifest.py` (tests in `tests/`). Each task gets a `timeout_seconds` from `timeout_minutes` (defaults: 60 min sessions, 180 min `converge`, integrator 60, gate 20 + soak per runtime target, or + 10 per dashboard / Genie space / table target, plus Genie probe question time).

In [0]:
%pip install --upgrade databricks-sdk
dbutils.library.restartPython()

In [0]:
import os

# Set by setup_build_waves.job.yml; fill build_job_id / smoke_job_id by hand for interactive runs
# (bundle summary shows them after deploy).
dbutils.widgets.text("build_root", "", "Build root (project Git folder)")
dbutils.widgets.text("manifest_path", "", "Manifest path (waves.json)")
dbutils.widgets.text("skills_dir", "", "Skills directory (absolute path)")
dbutils.widgets.text("skills_catalog", "", "UC catalog for skills")
dbutils.widgets.text("skills_schema", "", "UC schema for skills")
dbutils.widgets.text("runner_skill_id", "genie_build_session_runner", "Runner skill id")
dbutils.widgets.text("integrator_skill_id", "genie_build_wave_integrator", "Integrator skill id")
dbutils.widgets.text("build_job_id", "", "Build job id")
dbutils.widgets.text("smoke_job_id", "", "Smoke job id")
dbutils.widgets.text("gate_notebook_path", "", "Gate notebook deployed path")
dbutils.widgets.text("probe_notebook_path", "", "App probe notebook deployed path (default: next to the gate notebook)")
dbutils.widgets.text("placeholder_path", "", "Placeholder notebook deployed path")
dbutils.widgets.dropdown("create_skills_schema", "true", ["true", "false"], "Auto-create skills schema")
dbutils.widgets.dropdown("run_smoke", "false", ["true", "false"], "Run smoke job after patching")

build_root = dbutils.widgets.get("build_root").rstrip("/")
manifest_path = dbutils.widgets.get("manifest_path").strip()
skills_dir = dbutils.widgets.get("skills_dir").strip()
skills_catalog = dbutils.widgets.get("skills_catalog").strip()
skills_schema = dbutils.widgets.get("skills_schema").strip()
runner_skill_id = dbutils.widgets.get("runner_skill_id").strip()
integrator_skill_id = dbutils.widgets.get("integrator_skill_id").strip()
build_job_id = dbutils.widgets.get("build_job_id").strip()
smoke_job_id = dbutils.widgets.get("smoke_job_id").strip()
gate_notebook_path = dbutils.widgets.get("gate_notebook_path").strip()
probe_notebook_path = dbutils.widgets.get("probe_notebook_path").strip() or os.path.join(
    os.path.dirname(gate_notebook_path), "app_probe_check.ipynb")
placeholder_path = dbutils.widgets.get("placeholder_path").strip()
create_skills_schema = dbutils.widgets.get("create_skills_schema") == "true"
run_smoke = dbutils.widgets.get("run_smoke") == "true"

if not build_job_id or not smoke_job_id:
    raise ValueError(
        "build_job_id and smoke_job_id are required. "
        "Run `databricks bundle summary -o json` to find them after deploy."
    )

# Resolve manifest path: if relative, join with build_root
if not os.path.isabs(manifest_path):
    manifest_path = os.path.join(build_root, manifest_path)

print(f"build_root:      {build_root}")
print(f"manifest_path:   {manifest_path}")
print(f"skills_dir:      {skills_dir}")
print(f"skills:          {skills_catalog}.{skills_schema}")
print(f"build_job_id:    {build_job_id}")
print(f"smoke_job_id:    {smoke_job_id}")

In [0]:
import json

with open(manifest_path) as f:
    manifest = json.load(f)

SKILL_PARENT = f"schemas/{skills_catalog}.{skills_schema}"

# Skills to publish: key -> {skill_id, folder, fqn, files{relpath: text}}
SKILLS = {}
for key, skill_id in [("runner", runner_skill_id), ("integrator", integrator_skill_id)]:
    # Resolve folder: manifest may override, otherwise use skills_dir convention
    if manifest.get("skills", {}).get(key, {}).get("folder"):
        folder = os.path.join(build_root, manifest["skills"][key]["folder"])
    else:
        # Convention: skills_dir/<role-name>/
        role_dir = "genie-build-session-runner" if key == "runner" else "genie-build-wave-integrator"
        folder = os.path.join(skills_dir, role_dir)

    files = {}
    for dirpath, _, filenames in os.walk(folder):
        for fn in filenames:
            full = os.path.join(dirpath, fn)
            with open(full) as fh:
                files[os.path.relpath(full, folder)] = fh.read()
    if "SKILL.md" not in files:
        raise FileNotFoundError(f"{folder}/SKILL.md not found")

    fqn = f"{skills_catalog}.{skills_schema}.{skill_id}"
    SKILLS[key] = {"skill_id": skill_id, "folder": folder, "fqn": fqn, "files": files}
    print(f"  {key}: {fqn} ({len(files)} files from {folder})")

PLACEHOLDER = placeholder_path.rsplit("/", 1)[-1] if placeholder_path else "genie_task_placeholder"

In [0]:
# Fail fast on manifests the runtime can't execute safely.
# Errors stop the setup job; warnings are printed.
# waves_manifest.py, jobs_api.py and app_probes.py sit next to this notebook (cwd for
# workspace notebooks); waves_manifest and app_probes are shared with the gate notebook.
# Waves with "archived": true are validated but generate no tasks (see waves_manifest.py).
import sys
sys.path.insert(0, os.getcwd())
from app_probes import validate_probe_auth
from waves_manifest import is_archived, manifest_summary, validate_manifest

manifest_errors, manifest_warnings = validate_manifest(manifest)
for wmsg in manifest_warnings:
    print(f"  WARN: {wmsg}")

# probe_auth secrets must exist and be readable by this job (key names only; values are never read here).
from databricks.sdk import WorkspaceClient as _WC
_auths = [("probe_auth", manifest.get("probe_auth"))] + [
    (f"runtime_targets.{n}.probe_auth", t.get("probe_auth")) for n, t in (manifest.get("runtime_targets") or {}).items()]
_auths = [(where, a) for where, a in _auths if isinstance(a, dict) and not validate_probe_auth(a)]
if _auths:
    _w = _WC()
    for where, a in _auths:
        try:
            keys = {s.key for s in _w.secrets.list_secrets(a["secret_scope"])}
        except Exception as e:
            manifest_errors.append(f"{where}: secret scope {a['secret_scope']!r} not readable ({type(e).__name__})")
            continue
        for k in (a["client_id_key"], a["client_secret_key"]):
            if k not in keys:
                manifest_errors.append(f"{where}: key {k!r} not in secret scope {a['secret_scope']!r}")

if manifest_errors:
    raise ValueError("Invalid manifest:\n  - " + "\n  - ".join(manifest_errors))
print(manifest_summary(manifest))
_archived = [str(wv["wave_id"]) for wv in manifest["waves"] if is_archived(wv)]
if _archived:
    print(f"  Archived (no tasks): {', '.join(_archived)}")

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

schema_fqn = f"{skills_catalog}.{skills_schema}"

if create_skills_schema:
    try:
        w.schemas.get(schema_fqn)
        print(f"  Schema {schema_fqn} already exists")
    except Exception:
        print(f"  Creating schema {schema_fqn} ...")
        w.schemas.create(name=skills_schema, catalog_name=skills_catalog)
        print(f"  Schema {schema_fqn} created")
else:
    # Validate the schema exists even when not auto-creating
    try:
        w.schemas.get(schema_fqn)
        print(f"  Schema {schema_fqn} exists (create_skills_schema=false)")
    except Exception as e:
        raise RuntimeError(
            f"Schema {schema_fqn} does not exist and create_skills_schema is false. "
            f"Either create the schema manually or set create_skills_schema=true. Error: {e}"
        )

In [0]:
import io
from databricks.sdk.errors import NotFound, AlreadyExists
from databricks.sdk.service.catalog import Skill

# w = WorkspaceClient() — already created in the schema cell above
skills_published = {}

for key, sk in SKILLS.items():
    fqn = sk["fqn"]
    skill_name = f"skills/{fqn}"  # get/finalize need the skills/ prefix
    ok = False
    try:
        # 1. Create or verify skill exists (SDK method, not raw API)
        try:
            w.ai_gateway.get_skill(name=skill_name)
            print(f"  {key}: skill {fqn} already exists")
        except NotFound:
            try:
                w.ai_gateway.create_skill(
                    parent=SKILL_PARENT,
                    skill_id=sk["skill_id"],
                    skill=Skill(comment=f"Build wave {key} skill"),  # must be Skill dataclass, not dict
                )
                print(f"  {key}: created skill {fqn}")
            except AlreadyExists:
                print(f"  {key}: skill {fqn} already exists (race)")

        # 2. Upload files via the Files API (no upload_skill method exists)
        for relpath, text in sk["files"].items():
            upload_path = f"/Skills/{skills_catalog}/{skills_schema}/{sk['skill_id']}/{relpath}"
            w.files.upload(upload_path, io.BytesIO(text.encode("utf-8")), overwrite=True)

        # 3. Finalize (must use skills/ prefix in the name)
        w.ai_gateway.finalize_skill(name=skill_name)
        print(f"  {key}: published {fqn} ({len(sk['files'])} files)")
        ok = True
    except Exception as e:
        print(f"  {key}: WARN publish failed ({e}); will use workspace SKILL.md as fallback")
    skills_published[key] = ok

In [0]:
uid = w.current_user.me().id

existing = w.api_client.do(
    "GET",
    "/api/2.0/alerts-internal/scheduled-insights-list/GENIE_CODE",
    query={"parent_asset_name": f"users/{uid}"},
).get("scheduled_insights", [])

# --- Fix for duplicate automations on re-run ---
# The list endpoint may not return display_name, so also check what the build job
# already points to. If the job has genie_task configuration_ids from a prior setup,
# reuse those automations instead of creating new ones.
import jobs_api  # Jobs API 2.2 with full pagination (jobs with > 100 tasks)

_job_automation_names = set()
# Scan both the build job and the smoke job so smoke/capability automations are reused too.
# Read every page of tasks: an automation referenced only after task 100 would otherwise leak a duplicate.
for _jid in (build_job_id, smoke_job_id):
    try:
        _job_automation_names.update(jobs_api.iter_genie_configuration_ids(jobs_api.get_job_tasks(w.api_client, _jid)))
    except Exception:
        pass  # first deploy — job may still have placeholder

# Build a lookup from name -> automation for job-referenced automations
_job_automations_by_name = {e["name"]: e for e in existing if e.get("name") in _job_automation_names}
# Automations created via the API may be hidden from the list endpoint, so fetch any
# job-referenced automation that the list did not return.
for _name in _job_automation_names - set(_job_automations_by_name):
    try:
        _aut = w.api_client.do(
            "GET",
            f"/api/2.0/alerts-internal/scheduled-insights/{_name.rsplit('/', 1)[-1]}",
            query={"name": _name},
        )
        _aut = _aut.get("scheduled_insight", _aut)
        _aut.setdefault("name", _name)
        _aut.setdefault("id", _name.rsplit("/", 1)[-1])
        _job_automations_by_name[_name] = _aut
    except Exception as e:
        print(f"  WARN: could not read referenced automation {_name}: {e}")


def thin_prompt(role: str, key: str, params: list[str]) -> str:
    """Short automation prompt: load the UC Skill, fall back to the workspace SKILL.md."""
    sk = SKILLS[key]
    source = (
        f"Load and follow the `{sk['fqn']}` UC Skill exactly.\n"
        f"If you cannot load it, read and follow `{sk['folder']}/SKILL.md` instead.\n"
        if skills_published[key]
        else f"Read and follow `{sk['folder']}/SKILL.md` exactly.\n"
    )
    # The user_prompt must contain {{param_name}} placeholders — the Genie Code
    # runtime substitutes them from genie_task.parameters at runtime.
    param_lines = "\n".join(f"- {p}: {{{{{p}}}}}" for p in params)
    return f"{source}\nParameters:\n{param_lines}\n"


# Four automations: runner, integrator, smoke, capability
RUNNER_PARAMS = ["session_id", "wave", "run_id", "branch", "build_root", "manifest_path"]
INTEGRATOR_PARAMS = ["wave", "run_id", "branch", "build_root", "manifest_path"]
# Smoke nonce: a random hex string injected into the for_each inputs.
# The agent cannot guess it — if it echoes the correct value, we know
# {{input.*}} resolved in genie_task.parameters.
import uuid as _uuid
smoke_nonce = _uuid.uuid4().hex[:8]

SMOKE_PROMPT = (
    "You are a smoke test. Do not create, edit or delete any files or assets, "
    "and do not run any code or commands.\n"
    "Reply with exactly one line:\n"
    "smoke_ok session_id={{session_id}} nonce={{nonce}}"
)

# Capability probe: proves an unattended job task can do what converge sessions need
# (bundle CLI, apps/jobs read APIs, SQL on telemetry) without deploying anything.
CAPABILITY_PROMPT = (
    "You are a read-only capability probe for unattended build tasks. Do not create, edit or "
    "delete any files or assets, and do not run bundle deploy, bundle run, or any command that "
    "changes state.\n"
    "Run each check and report one line per check as name=ok or name=fail:<short reason>:\n"
    "1. cli_bundle: Databricks CLI `bundle validate --target {{bundle_target}}` from {{bundle_root}}\n"
    "2. apps_read: list or get one Databricks App (CLI `apps list` or SDK w.apps.list)\n"
    "3. jobs_read: get job {{smoke_job_id}} (CLI `jobs get` or SDK)\n"
    "4. sql_read: run `SELECT 1` through SQL\n"
    "5. telemetry_read: if {{logs_table}} is not empty, run `SELECT count(*) FROM {{logs_table}} "
    "WHERE date >= current_date() - 1`; otherwise report telemetry_read=skipped\n"
    "6. deploy_permitted: would your tools allow `bundle deploy` here? Answer yes, no or unknown; do NOT run it\n"
    "Last line exactly: capability_ok nonce={{nonce}}"
)

automation_configs = [
    ("runner", "runner", thin_prompt("runner", "runner", RUNNER_PARAMS)),
    ("integrator", "integrator", thin_prompt("integrator", "integrator", INTEGRATOR_PARAMS)),
    ("smoke", None, SMOKE_PROMPT),
    ("capability", None, CAPABILITY_PROMPT),
]


def _prompt_of(aut: dict) -> str:
    return aut.get("user_prompt", aut.get("prompt", "")) or ""


def _role_of(aut: dict) -> str | None:
    """Identify which role an existing automation plays from its prompt text."""
    p = _prompt_of(aut)
    if "capability_ok" in p:
        return "capability"
    if "smoke_ok" in p:
        return "smoke"
    if "session_id" in p:
        return "runner"
    if "wave" in p:
        return "integrator"
    return None

configuration_ids = {}
for auto_key, skill_key, prompt in automation_configs:
    # Find existing by matching the key in the display_name
    display_name = f"genie-build-waves:{auto_key}"
    # Try matching by display_name first, then by name from the job's existing tasks
    match = next((e for e in existing if e.get("display_name") == display_name), None)
    if not match:
        # Reuse an automation the build or smoke job already references (display_name may be unset)
        match = next((aut for aut in _job_automations_by_name.values() if _role_of(aut) == auto_key), None)

    if match:
        # Update via PATCH with etag (optimistic concurrency)
        iid = match["id"]
        full_name = match["name"]
        etag = match.get("etag") or w.api_client.do(
            "GET",
            f"/api/2.0/alerts-internal/scheduled-insights/{iid}",
            query={"name": full_name},
        )["etag"]
        w.api_client.do("PATCH", f"/api/2.0/alerts-internal/scheduled-insights/{iid}", body={
            "insight_id": iid,
            "update_mask": "user_prompt,display_name",
            "etag": etag,
            "scheduled_insight": {
                "name": full_name,
                "display_name": display_name,
                "user_prompt": prompt,
            },
        })
        # genie_task.configuration_id expects the full name, not just the hex ID
        configuration_ids[auto_key] = full_name
        print(f"  {auto_key}: updated automation {full_name}")
    else:
        # Create via POST — parent_asset_name at top level, rest in scheduled_insight
        resp = w.api_client.do("POST", "/api/2.0/alerts-internal/scheduled-insights", body={
            "parent_asset_name": f"users/{uid}",
            "scheduled_insight": {
                "display_name": display_name,
                "insight_type": "GENIE_CODE",
                "user_prompt": prompt,
            },
        })
        # Response may nest the insight; extract the full name for genie_task.configuration_id
        si = resp.get("scheduled_insight", resp)
        full_name = si.get("name", "")
        if not full_name:
            # Fallback: construct name from ID
            new_id = si.get("id", "")
            full_name = f"genie_code/users/{uid}/scheduled_insights/{new_id}" if new_id else ""
        if not full_name:
            print(f"  {auto_key}: WARNING could not extract name from response: {json.dumps(resp, indent=2)}")
        configuration_ids[auto_key] = full_name
        print(f"  {auto_key}: created automation {full_name}")

print(f"\nconfiguration_ids: {json.dumps(configuration_ids, indent=2)}")

In [0]:
# Task generation lives in waves_manifest.py (tested in tests/). Per wave:
# w<wave_id>_sessions (for_each) -> _integrate -> _gate -> _gate_check (If/else).
# Archived waves generate no tasks; the first runnable wave has no upstream dependency.
from waves_manifest import genie_task_retries, generate_wave_tasks, make_genie_task, make_notebook_task, runnable_waves

# Task-level retries for Genie Code tasks. Agent start-up fails transiently
# (INTERNAL_ERROR: start_agent_run); a retry is cheaper than a repair or a fresh run.
# Sessions and the integrator are safe to repeat: sessions rewrite their own status
# file and the integrator stages by explicit path (nothing staged = no commit).
# Timeouts are not retried: a timed-out converge loop should surface, not restart.
# Override per manifest with "task_retries": {"max_retries": N, "min_retry_interval_seconds": S}.
GENIE_TASK_RETRIES = genie_task_retries(manifest)
print(f"Genie task retries: {GENIE_TASK_RETRIES}")

build_tasks = generate_wave_tasks(manifest, configuration_ids, gate_notebook_path, retries=GENIE_TASK_RETRIES)
_runnable = runnable_waves(manifest)
print(f"Generated {len(build_tasks)} tasks across {len(_runnable)} runnable waves "
      f"({len(manifest['waves']) - len(_runnable)} archived skipped):")
for t in build_tasks:
    dep = [d["task_key"] for d in t.get("depends_on", [])]
    inner = (t.get("for_each_task") or {}).get("task") or {}
    tmo = t.get("timeout_seconds") or inner.get("timeout_seconds")
    conc = (t.get("for_each_task") or {}).get("concurrency")
    print(f"  {t['task_key']}" + (f" (depends: {dep})" if dep else "")
          + (f" [concurrency {conc}]" if conc else "") + (f" [timeout {tmo // 60} min]" if tmo else ""))

In [0]:
import jobs_api


def reset_job_tasks(job_id, new_tasks, parameters=None):
    """Replace a job's tasks entirely via jobs/reset (Jobs API 2.2), preserving all other settings.

    API 2.1 rejects jobs with more than 100 tasks. The current settings are read with
    every jobs/get page (next_page_token), so nothing past the first 100 elements is lost.
    If parameters is provided, also update the job-level parameter defaults so that
    repair_run (which falls back to job defaults when job_parameters is not passed)
    uses the correct branch/build_root/manifest_path.
    """
    jobs_api.reset_job_tasks(w.api_client, job_id, new_tasks, parameters=parameters)
    # Read back (all pages) and confirm the job now has exactly the generated task keys.
    got = [t["task_key"] for t in jobs_api.get_job_tasks(w.api_client, job_id)]
    want = [t["task_key"] for t in new_tasks]
    if sorted(got) != sorted(want):
        raise RuntimeError(f"job {job_id}: reset wrote {len(want)} tasks but jobs/get returns {len(got)}; "
                           f"missing={sorted(set(want) - set(got))[:10]} extra={sorted(set(got) - set(want))[:10]}")


# --- Reset the build job ---
# Also update job-level parameter defaults so repair_run (which falls back to
# job defaults when job_parameters is omitted) uses the correct values.
build_params = [
    {"name": "branch", "default": manifest.get("branch", "")},
    {"name": "build_root", "default": build_root},
    {"name": "manifest_path", "default": manifest_path},
]
reset_job_tasks(build_job_id, build_tasks, parameters=build_params)
print(f"Build job {build_job_id}: reset with {len(build_tasks)} tasks and updated parameter defaults")

# --- Reset the smoke job ---
# Inject the nonce into each input. If the agent echoes it back, {{input.*}} works.
smoke_inputs = json.dumps([
    {"session_id": "A", "nonce": smoke_nonce},
    {"session_id": "B", "nonce": smoke_nonce},
])
print(f"Smoke nonce: {smoke_nonce} (agent must echo this back to prove parameter resolution)")

smoke_tasks = [
    {
        "task_key": "smoke_sessions",
        "for_each_task": {
            "inputs": smoke_inputs,
            "concurrency": 2,
            "task": make_genie_task(
                task_key="smoke_session",
                config_id=configuration_ids["smoke"],
                params={
                    "session_id": "{{input.session_id}}",
                    "nonce": "{{input.nonce}}",
                },
                retries=GENIE_TASK_RETRIES,
            ),
        },
    }
]

# Capability probe runs alongside the for_each smoke. It targets the first runtime target
# (if any) so it exercises the same bundle + telemetry a converge session would use.
_rt = next(iter((manifest.get("runtime_targets") or {}).values()), {})
capability_params = {
    "nonce": smoke_nonce,
    "bundle_root": _rt.get("bundle_root") or build_root,
    "bundle_target": _rt.get("bundle_target") or manifest.get("bundle_target") or "dev",
    "smoke_job_id": smoke_job_id,
    "logs_table": (_rt.get("telemetry") or {}).get("logs_table", ""),
}
smoke_tasks.append(
    make_genie_task(
        task_key="capability_probe",
        config_id=configuration_ids["capability"],
        params=capability_params,
        timeout_minutes=30,
        retries=GENIE_TASK_RETRIES,
    )
)
# Deterministic access check: proves the probe_auth service principal can make authenticated app
# requests (gate: probes, min_traces), that trace sources are readable, and that the job can read
# each Genie space's configuration (gate: genie_space checks need CAN_EDIT).
smoke_tasks.append(
    make_notebook_task(
        task_key="app_probe_check",
        notebook_path=probe_notebook_path,
        params={"manifest_path": manifest_path, "build_root": build_root},
        timeout_minutes=15,
    )
)
reset_job_tasks(smoke_job_id, smoke_tasks)
print(f"Smoke job {smoke_job_id}: reset with {len(smoke_tasks)} tasks")

patched = {"build": len(build_tasks), "smoke": len(smoke_tasks)}

In [0]:
import time

smoke_result = None
if not run_smoke:
    print("run_smoke=false: skipping. Start the build job from the Jobs UI when ready.")
else:
    resp = jobs_api.run_now(w.api_client, smoke_job_id)
    run_id_smoke = resp["run_id"]
    print(f"Started smoke run {run_id_smoke}: {w.config.host}/jobs/{smoke_job_id}/runs/{run_id_smoke}")

    deadline = time.time() + 30 * 60
    while True:
        run = jobs_api.get_run(w.api_client, run_id_smoke)  # all pages of tasks
        state = run.get("state", {})
        life = state.get("life_cycle_state", "")
        if life in ("TERMINATED", "SKIPPED", "INTERNAL_ERROR"):
            break
        if time.time() > deadline:
            print("WARN: smoke timed out after 30 min")
            break
        time.sleep(15)

    result_state = state.get("result_state", "UNKNOWN")
    print(f"Smoke run {run_id_smoke}: {result_state}")

    # Check iterations
    iterations = []
    for task in run.get("tasks", []):
        if task.get("for_each_task"):
            stats = task["for_each_task"].get("stats", {})
            iterations = task["for_each_task"].get("completed_iterations", [])
            print(f"  for_each stats: {stats}")
            print(f"  iterations: {len(iterations)}")

    task_states = {
        t.get("task_key"): t.get("state", {}).get("result_state", t.get("state", {}).get("life_cycle_state"))
        for t in run.get("tasks", [])
    }
    print(f"  task states: {task_states}")
    print(
        f"  Check the capability_probe output for cli_bundle / apps_read / jobs_read / sql_read / "
        f"telemetry_read / deploy_permitted and 'capability_ok nonce={smoke_nonce}'. "
        f"Any fail there means converge sessions will not work unattended."
    )

    # Deterministic app probe output (identity + per-app pass/fail/skipped)
    app_probe = None
    probe_task = next((t for t in run.get("tasks", []) if t.get("task_key") == "app_probe_check"), None)
    if probe_task:
        try:
            out = jobs_api.get_run_output(w.api_client, probe_task["run_id"])
            nb = (out.get("notebook_output") or {}).get("result")
            app_probe = json.loads(nb) if nb else {"status": "fail", "error": out.get("error", "")[:500]}
        except Exception as e:
            app_probe = {"status": "unknown", "error": str(e)[:300]}
        print(f"  app_probe_check: {app_probe.get('status')} as {app_probe.get('identity', '?')}")
        for n, r in (app_probe.get("targets") or {}).items():
            print(f"    {n} ({r.get('app') or r.get('space')}): {r.get('result')} {r.get('detail') or r.get('problems') or ''}")
        if app_probe.get("status") == "skipped":
            print("    No running app to probe: auth for probes/min_traces is NOT yet proven. Re-run the smoke once the app is up.")

    smoke_result = {
        "run_id": run_id_smoke,
        "result_state": result_state,
        "iteration_count": len(iterations),
        "task_states": task_states,
        "nonce": smoke_nonce,
        "app_probe": app_probe,
    }

In [0]:
result = {
    "build_job_id": build_job_id,
    "smoke_job_id": smoke_job_id,
    "skills_published": {k: (SKILLS[k]["fqn"] if v else False) for k, v in skills_published.items()},
    "configuration_ids": configuration_ids,
    "patched": patched,
    "smoke_result": smoke_result,
}
print(json.dumps(result, indent=2))
dbutils.notebook.exit(json.dumps(result))